# 🔑 GSL Capa Administrativa — Manifold de Comportamiento
**Active Directory · Azure AD · Linux auditd · IAM genérico**

## Principio
Los otros dominios observan **efectos**.
La capa administrativa observa **intenciones declaradas**.
Cada acción queda registrada con autor, timestamp y objeto afectado.
El manifold no juzga si la acción fue válida — juzga si pertenece al patrón.

## Dimensiones únicas de este manifold
```
1. Secuencia lógica de cambios  — orden topológico del grafo de permisos
2. Ventana temporal sensible    — cuándo ocurren cambios de alto impacto
3. Coherencia actor-objeto      — quién modifica qué
4. Duración de privilegio elev. — cuánto tiempo se mantiene acceso elevado
5. Correlación entre capas      — sincronización con nodo físico / GSL M1
6. Autoprotección del GSL       — detecta cambios que apuntan al propio sistema
```

## Integración con el sistema GSL
- Mismo contrato que el Resistor y el nodo físico
- Eventos `admin_anomaly` al PolicyAdapter del Modo 2
- Único módulo que detecta ataques contra el sistema de detección mismo

## Fuentes soportadas
```
Active Directory  → Event Log (4720, 4722, 4728, 4732, 4756...)
Azure AD          → Audit Logs (addMember, updatePolicy...)
Linux auditd      → ausearch / aureport output
IAM genérico      → JSON / CSV con campos mínimos
```

## 1. Instalación de dependencias

In [ ]:
!pip install gradio numpy pandas matplotlib plotly networkx requests -q
print('✅ Dependencias listas.')

## 2. Configuración y taxonomía de eventos administrativos

El esquema mínimo común normaliza eventos de cualquier plataforma IAM
a un conjunto de campos interpretables por el motor de manifold.

In [ ]:
import json, uuid, hashlib, time, copy
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
from collections import defaultdict, deque

# ─── Taxonomía de eventos administrativos ────────────────────
# Cada evento tiene: categoría, impacto, reversibilidad, y si
# puede afectar al propio sistema de detección (GSL_RELEVANT)

EVENT_TAXONOMY = {
    # Gestión de cuentas
    'user_created':          {'cat': 'account',    'impact': 'medium', 'rev': True,  'gsl': False},
    'user_deleted':          {'cat': 'account',    'impact': 'high',   'rev': False, 'gsl': False},
    'user_enabled':          {'cat': 'account',    'impact': 'medium', 'rev': True,  'gsl': False},
    'user_disabled':         {'cat': 'account',    'impact': 'medium', 'rev': True,  'gsl': False},
    'password_reset':        {'cat': 'account',    'impact': 'medium', 'rev': True,  'gsl': False},
    'mfa_disabled':          {'cat': 'account',    'impact': 'high',   'rev': True,  'gsl': False},
    # Gestión de permisos
    'permission_granted':    {'cat': 'permission', 'impact': 'high',   'rev': True,  'gsl': True},
    'permission_revoked':    {'cat': 'permission', 'impact': 'medium', 'rev': True,  'gsl': True},
    'role_assigned':         {'cat': 'permission', 'impact': 'high',   'rev': True,  'gsl': False},
    'role_removed':          {'cat': 'permission', 'impact': 'medium', 'rev': True,  'gsl': False},
    'admin_role_granted':    {'cat': 'permission', 'impact': 'critical','rev': True, 'gsl': True},
    'admin_role_revoked':    {'cat': 'permission', 'impact': 'medium', 'rev': True,  'gsl': True},
    # Grupos
    'group_member_added':    {'cat': 'group',      'impact': 'medium', 'rev': True,  'gsl': False},
    'group_member_removed':  {'cat': 'group',      'impact': 'low',    'rev': True,  'gsl': False},
    'group_created':         {'cat': 'group',      'impact': 'low',    'rev': True,  'gsl': False},
    'group_deleted':         {'cat': 'group',      'impact': 'medium', 'rev': False, 'gsl': False},
    # Políticas
    'policy_created':        {'cat': 'policy',     'impact': 'high',   'rev': True,  'gsl': True},
    'policy_modified':       {'cat': 'policy',     'impact': 'high',   'rev': True,  'gsl': True},
    'policy_deleted':        {'cat': 'policy',     'impact': 'critical','rev': False,'gsl': True},
    'audit_log_cleared':     {'cat': 'policy',     'impact': 'critical','rev': False,'gsl': True},
    'audit_policy_disabled': {'cat': 'policy',     'impact': 'critical','rev': True, 'gsl': True},
    # Sesiones privilegiadas
    'sudo_session_start':    {'cat': 'session',    'impact': 'high',   'rev': True,  'gsl': True},
    'sudo_session_end':      {'cat': 'session',    'impact': 'low',    'rev': True,  'gsl': False},
    'runas_admin':           {'cat': 'session',    'impact': 'high',   'rev': True,  'gsl': True},
    'service_account_login': {'cat': 'session',    'impact': 'medium', 'rev': True,  'gsl': False},
    # Recursos
    'resource_shared':       {'cat': 'resource',   'impact': 'medium', 'rev': True,  'gsl': False},
    'acl_modified':          {'cat': 'resource',   'impact': 'high',   'rev': True,  'gsl': True},
    'firewall_rule_added':   {'cat': 'resource',   'impact': 'high',   'rev': True,  'gsl': True},
    'firewall_rule_deleted': {'cat': 'resource',   'impact': 'critical','rev': False,'gsl': True},
}

IMPACT_WEIGHTS = {'low': 0.1, 'medium': 0.3, 'high': 0.6, 'critical': 1.0}

# Secuencia lógica esperada (grafo de precondiciones)
# B requiere que A haya ocurrido antes (en el contexto del mismo actor/objeto)
LOGICAL_SEQUENCE = {
    'permission_granted': ['user_created', 'user_enabled'],
    'admin_role_granted': ['role_assigned', 'permission_granted'],
    'acl_modified':       ['permission_granted', 'resource_shared'],
    'sudo_session_start': ['user_enabled', 'role_assigned'],
}

# Horarios normales de cambios administrativos por tipo
NORMAL_WINDOWS = {
    'account':    {'hours': (8, 18),  'days': (0, 4)},   # lun-vie, horario laboral
    'permission': {'hours': (9, 17),  'days': (0, 4)},   # más restrictivo
    'policy':     {'hours': (10, 16), 'days': (0, 4)},   # solo en horas pico
    'session':    {'hours': (0, 23),  'days': (0, 6)},   # cualquier hora
    'group':      {'hours': (8, 18),  'days': (0, 4)},
    'resource':   {'hours': (9, 17),  'days': (0, 4)},
}

# ─── Config del módulo ────────────────────────────────────────
DEFAULT_ADMIN_CONFIG = {
    'org_name':         'Demo GSL Admin',
    'principal_id':     'ORG-DEMO-001',
    'iam_platform':     'generic',   # 'ad' | 'azure_ad' | 'auditd' | 'generic'
    'mode2_webhook_url': '',
    'gsl_protected_resources': [
        'gsl_service_account', 'gsl_policy', 'gsl_audit_log',
        'gsl_config', 'resistor_rules', 'manifold_store'
    ],
    'thresholds': {
        'dissonance_alert':       0.28,
        'dissonance_critical':    0.55,
        'privilege_window_max_s': 3600,   # máx 1h de privilegio elevado normal
        'off_hours_weight':       2.5,    # multiplicador para cambios fuera de horario
        'sequence_violation_w':   3.0,    # multiplicador para secuencia lógica violada
        'gsl_self_attack_w':      5.0,    # multiplicador si objeto es recurso GSL
    },
    'admins': [
        {'id': 'adm01', 'name': 'Admin Principal',  'role': 'sysadmin',
         'typical_hours': [9, 18], 'manages': ['account','permission','policy']},
        {'id': 'adm02', 'name': 'Ops Seguridad',    'role': 'secadmin',
         'typical_hours': [8, 20], 'manages': ['permission','policy','resource']},
        {'id': 'adm03', 'name': 'Helpdesk',         'role': 'helpdesk',
         'typical_hours': [9, 17], 'manages': ['account','group']},
        {'id': 'svc01', 'name': 'Service Account',  'role': 'service',
         'typical_hours': [0, 23], 'manages': ['session']},
    ]
}


def load_admin_config(path: str = 'admin_config.json') -> dict:
    p = Path(path)
    if p.exists():
        with open(p) as f:
            custom = json.load(f)
        merged = {**DEFAULT_ADMIN_CONFIG, **custom}
        for k in ('thresholds',):
            if k in DEFAULT_ADMIN_CONFIG and k in custom:
                merged[k] = {**DEFAULT_ADMIN_CONFIG[k], **custom[k]}
        print(f'✅ Config cargada: {merged["org_name"]}')
    else:
        merged = copy.deepcopy(DEFAULT_ADMIN_CONFIG)
        print(f'ℹ️  Sin admin_config.json — usando defaults')
    return merged


ACFG    = load_admin_config()
RUN_ID  = f"GSL-ADM-{ACFG['principal_id']}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
ARTIFACTS = Path('gsl_admin_artifacts') / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f'  Run ID    : {RUN_ID}')
print(f'  Plataforma: {ACFG["iam_platform"]}')
print(f'  Admins    : {len(ACFG["admins"])}')
print(f'  Recursos GSL protegidos: {len(ACFG["gsl_protected_resources"])}')

## 3. Parsers de fuentes IAM y generador sintético

Normaliza eventos de AD, Azure AD, auditd, o CSV genérico
al esquema mínimo común del módulo administrativo.

In [ ]:
# ─── Esquema mínimo de evento administrativo ─────────────────
@dataclass
class AdminEvent:
    event_id:    str
    timestamp:   float          # unix timestamp
    actor_id:    str            # quién ejecutó la acción
    actor_role:  str
    event_type:  str            # del EVENT_TAXONOMY
    category:    str
    object_id:   str            # sobre qué recurso/usuario
    object_type: str            # 'user'|'group'|'policy'|'resource'|'gsl'
    impact:      str
    source_ip:   str
    success:     bool
    raw_code:    str = ''       # código original de la plataforma (ej: '4728')
    platform:    str = 'generic'


# ─── Parsers por plataforma ───────────────────────────────────

# Mapa de Event ID de Windows AD → event_type GSL
AD_EVENT_MAP = {
    '4720': 'user_created',       '4726': 'user_deleted',
    '4722': 'user_enabled',       '4725': 'user_disabled',
    '4723': 'password_reset',     '4724': 'password_reset',
    '4728': 'group_member_added', '4729': 'group_member_removed',
    '4732': 'group_member_added', '4733': 'group_member_removed',
    '4756': 'group_member_added', '4757': 'group_member_removed',
    '4670': 'permission_granted', '4907': 'acl_modified',
    '4719': 'audit_policy_disabled', '1102': 'audit_log_cleared',
    '4672': 'admin_role_granted', '4673': 'runas_admin',
}

# Mapa de operaciones Azure AD
AZURE_EVENT_MAP = {
    'Add user':                  'user_created',
    'Delete user':               'user_deleted',
    'Update user':               'user_enabled',
    'Reset user password':       'password_reset',
    'Add member to role':        'admin_role_granted',
    'Remove member from role':   'admin_role_revoked',
    'Add member to group':       'group_member_added',
    'Remove member from group':  'group_member_removed',
    'Update policy':             'policy_modified',
    'Add policy':                'policy_created',
    'Delete policy':             'policy_deleted',
    'Update conditional access': 'policy_modified',
}

# Mapa de syscalls auditd
AUDITD_MAP = {
    'useradd':   'user_created',   'userdel':  'user_deleted',
    'usermod':   'user_enabled',   'passwd':   'password_reset',
    'groupadd':  'group_created',  'groupdel': 'group_deleted',
    'chown':     'acl_modified',   'chmod':    'permission_granted',
    'setfacl':   'permission_granted', 'sudo': 'sudo_session_start',
    'auditctl':  'audit_policy_disabled',
}


def parse_ad_event(raw: dict, cfg: dict) -> Optional[AdminEvent]:
    event_code = str(raw.get('EventID', raw.get('event_id', '')))
    etype = AD_EVENT_MAP.get(event_code)
    if not etype:
        return None
    tax = EVENT_TAXONOMY.get(etype, {})
    obj = raw.get('TargetUserName', raw.get('ObjectName', 'unknown'))
    obj_type = 'gsl' if obj in cfg.get('gsl_protected_resources', []) else 'user'
    return AdminEvent(
        event_id   = str(uuid.uuid4())[:8],
        timestamp  = pd.to_datetime(raw.get('TimeCreated', raw.get('timestamp', ''))).timestamp(),
        actor_id   = raw.get('SubjectUserName', raw.get('actor', 'unknown')),
        actor_role = raw.get('SubjectDomainName', 'domain'),
        event_type = etype, category = tax.get('cat', 'unknown'),
        object_id  = obj, object_type = obj_type,
        impact     = tax.get('impact', 'low'),
        source_ip  = raw.get('IpAddress', raw.get('source_ip', '0.0.0.0')),
        success    = raw.get('Keywords', 'Audit Success') == 'Audit Success',
        raw_code   = event_code, platform = 'ad',
    )


def parse_azure_event(raw: dict, cfg: dict) -> Optional[AdminEvent]:
    op = raw.get('operationName', raw.get('Operation', ''))
    etype = next((v for k, v in AZURE_EVENT_MAP.items()
                  if k.lower() in op.lower()), None)
    if not etype:
        return None
    tax = EVENT_TAXONOMY.get(etype, {})
    obj = raw.get('targetResources', [{}])
    obj = obj[0].get('displayName', 'unknown') if obj else 'unknown'
    obj_type = 'gsl' if obj in cfg.get('gsl_protected_resources', []) else 'resource'
    ts_raw = raw.get('activityDateTime', raw.get('createdDateTime', ''))
    try:
        ts = pd.to_datetime(ts_raw, utc=True).timestamp()
    except:
        ts = time.time()
    return AdminEvent(
        event_id   = raw.get('id', str(uuid.uuid4())[:8]),
        timestamp  = ts,
        actor_id   = raw.get('initiatedBy', {}).get('user', {}).get('userPrincipalName', 'unknown'),
        actor_role = raw.get('initiatedBy', {}).get('app', {}).get('displayName', 'user'),
        event_type = etype, category = tax.get('cat', 'unknown'),
        object_id  = obj, object_type = obj_type,
        impact     = tax.get('impact', 'low'),
        source_ip  = raw.get('ipAddress', '0.0.0.0'),
        success    = raw.get('result', 'success') == 'success',
        raw_code   = op[:40], platform = 'azure_ad',
    )


def parse_auditd_event(raw: dict, cfg: dict) -> Optional[AdminEvent]:
    syscall = raw.get('syscall', raw.get('exe', '').split('/')[-1])
    etype   = AUDITD_MAP.get(syscall)
    if not etype:
        return None
    tax = EVENT_TAXONOMY.get(etype, {})
    obj = raw.get('name', raw.get('comm', raw.get('exe', 'unknown')))
    obj_type = 'gsl' if any(
        r in str(obj) for r in cfg.get('gsl_protected_resources', [])
    ) else 'resource'
    return AdminEvent(
        event_id   = raw.get('serial', str(uuid.uuid4())[:8]),
        timestamp  = float(raw.get('time', time.time())),
        actor_id   = raw.get('auid', raw.get('uid', 'unknown')),
        actor_role = 'unix_user',
        event_type = etype, category = tax.get('cat', 'unknown'),
        object_id  = str(obj), object_type = obj_type,
        impact     = tax.get('impact', 'low'),
        source_ip  = raw.get('addr', '127.0.0.1'),
        success    = raw.get('success', 'yes') == 'yes',
        raw_code   = syscall, platform = 'auditd',
    )


def parse_generic_csv(df: pd.DataFrame, cfg: dict) -> List[AdminEvent]:
    col_map = {
        'timestamp': ['timestamp','time','datetime','date'],
        'actor_id':  ['actor','user','admin','operator','account'],
        'event_type':['event_type','action','operation','event'],
        'object_id': ['object','target','resource','subject'],
        'source_ip': ['ip','source_ip','src_ip','ipaddress'],
        'success':   ['success','result','status','outcome'],
    }
    cols_lower = {c.lower(): c for c in df.columns}
    mapped = {}
    for field, candidates in col_map.items():
        for cand in candidates:
            if cand in cols_lower:
                mapped[field] = cols_lower[cand]
                break

    events = []
    for _, row in df.iterrows():
        etype_raw = str(row.get(mapped.get('event_type', ''), 'unknown'))
        # Buscar coincidencia en taxonomy
        etype = next((k for k in EVENT_TAXONOMY
                      if k in etype_raw.lower()), 'permission_granted')
        tax   = EVENT_TAXONOMY.get(etype, {})
        obj   = str(row.get(mapped.get('object_id', ''), 'unknown'))
        obj_type = 'gsl' if obj in cfg.get('gsl_protected_resources', []) else 'resource'
        try:
            ts = pd.to_datetime(row.get(mapped.get('timestamp',''), '')).timestamp()
        except:
            ts = time.time()
        suc_raw = str(row.get(mapped.get('success',''), 'true')).lower()
        events.append(AdminEvent(
            event_id   = str(uuid.uuid4())[:8],
            timestamp  = ts,
            actor_id   = str(row.get(mapped.get('actor_id',''), 'unknown')),
            actor_role = 'user',
            event_type = etype, category = tax.get('cat', 'unknown'),
            object_id  = obj, object_type = obj_type,
            impact     = tax.get('impact', 'low'),
            source_ip  = str(row.get(mapped.get('source_ip',''), '0.0.0.0')),
            success    = suc_raw in ('true','success','1','yes'),
            platform   = 'generic',
        ))
    return events


def parse_admin_log(data: Any, platform: str, cfg: dict) -> List[AdminEvent]:
    """Router principal — detecta plataforma y aplica el parser correcto."""
    if platform == 'ad':
        raw_list = data if isinstance(data, list) else [data]
        return [e for r in raw_list
                for e in [parse_ad_event(r, cfg)] if e]
    elif platform == 'azure_ad':
        raw_list = data if isinstance(data, list) else [data]
        return [e for r in raw_list
                for e in [parse_azure_event(r, cfg)] if e]
    elif platform == 'auditd':
        raw_list = data if isinstance(data, list) else [data]
        return [e for r in raw_list
                for e in [parse_auditd_event(r, cfg)] if e]
    else:
        df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        return parse_generic_csv(df, cfg)


# ─── Generador sintético ──────────────────────────────────────

def generate_admin_events(
    cfg:       dict,
    n_events:  int   = 200,
    scenario:  str   = 'normal',
    seed:      int   = 42,
) -> List[AdminEvent]:
    """
    Genera eventos administrativos sintéticos.

    Escenarios:
    'normal'         — operación rutinaria
    'priv_escalation'— escalada de privilegios (secuencia invertida)
    'off_hours'      — cambios críticos fuera de horario
    'gsl_attack'     — intento de deshabilitar el GSL
    'slow_recon'     — reconocimiento lento (muchas cuentas consultadas)
    'mixed'          — mezcla de normal + ataque
    """
    import random
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    admins  = cfg['admins']
    gsl_res = cfg['gsl_protected_resources']
    base    = datetime(2025, 1, 20, 9, 0, 0, tzinfo=timezone.utc)
    events  = []

    normal_types = [
        ('user_created','adm03'), ('user_enabled','adm03'),
        ('password_reset','adm03'), ('group_member_added','adm03'),
        ('permission_granted','adm01'), ('role_assigned','adm01'),
        ('policy_modified','adm02'), ('sudo_session_start','svc01'),
        ('sudo_session_end','svc01'), ('resource_shared','adm02'),
    ]

    def make_event(etype, actor_id, ts, obj_id=None, obj_type='user',
                   src_ip=None, success=True):
        adm = next((a for a in admins if a['id'] == actor_id), admins[0])
        tax = EVENT_TAXONOMY.get(etype, {})
        return AdminEvent(
            event_id   = str(uuid.uuid4())[:8],
            timestamp  = ts.timestamp(),
            actor_id   = actor_id,
            actor_role = adm['role'],
            event_type = etype,
            category   = tax.get('cat', 'unknown'),
            object_id  = obj_id or f'obj_{rng.randint(1,20):03d}',
            object_type= obj_type,
            impact     = tax.get('impact', 'low'),
            source_ip  = src_ip or f'192.168.1.{rng.randint(10,50)}',
            success    = success,
            platform   = 'generic',
        )

    # ── Eventos normales (siempre presentes) ──
    n_normal = n_events if scenario == 'normal' else n_events // 2
    for i in range(n_normal):
        etype, actor = rng.choice(normal_types)
        adm = next((a for a in admins if a['id'] == actor), admins[0])
        h = rng.randint(*adm['typical_hours'])
        day_off = rng.randint(0, 59)
        ts = base.replace(hour=h, minute=rng.randint(0,59)) + timedelta(days=day_off)
        events.append(make_event(etype, actor, ts))

    # ── Escenarios de ataque ──────────────────
    attack_ts = base.replace(hour=rng.randint(2, 5)) + timedelta(days=30)

    if scenario in ('priv_escalation', 'mixed'):
        # Escalada: admin_role_granted sin precondiciones
        # Secuencia invertida: permisos antes de cuenta
        events.append(make_event('admin_role_granted', 'adm01',
            attack_ts, 'compromised_user', 'user',
            src_ip='203.45.67.89'))
        events.append(make_event('permission_granted', 'adm01',
            attack_ts + timedelta(minutes=2), 'compromised_user', 'user',
            src_ip='203.45.67.89'))
        events.append(make_event('user_created', 'adm01',
            attack_ts + timedelta(minutes=5), 'compromised_user', 'user',
            src_ip='203.45.67.89'))  # ← creado DESPUÉS de recibir permisos

    if scenario in ('off_hours', 'mixed'):
        # Cambios críticos a las 3 am
        for etype in ['policy_modified', 'acl_modified', 'mfa_disabled']:
            events.append(make_event(etype, 'adm02',
                attack_ts + timedelta(minutes=rng.randint(1,30)),
                src_ip='10.0.0.99'))

    if scenario in ('gsl_attack', 'mixed'):
        # Intento de deshabilitar el sistema de detección
        for res in gsl_res[:3]:
            events.append(make_event('permission_revoked', 'adm01',
                attack_ts, res, 'gsl', src_ip='185.220.101.45'))
        events.append(make_event('audit_policy_disabled', 'adm01',
            attack_ts + timedelta(minutes=1), 'audit_policy', 'policy',
            src_ip='185.220.101.45'))
        events.append(make_event('firewall_rule_deleted', 'adm01',
            attack_ts + timedelta(minutes=3), 'resistor_rules', 'gsl',
            src_ip='185.220.101.45'))

    if scenario == 'slow_recon':
        # Reconocimiento lento: muchos accesos a cuentas distintas
        for i in range(30):
            ts_r = base + timedelta(days=i*2, hours=rng.randint(10,15))
            events.append(make_event('permission_granted', 'adm01',
                ts_r, f'recon_target_{i:02d}', 'user'))

    events.sort(key=lambda e: e.timestamp)
    return events


print('✅ Parsers IAM y generador sintético listos.')
print(f'   Plataformas: AD · Azure AD · auditd · CSV genérico')
print(f'   Tipos de evento: {len(EVENT_TAXONOMY)}')

## 4. Motor de manifold administrativo

Seis dimensiones únicas de este dominio.
El grafo de permisos captura la topología de las relaciones.
La detección de auto-ataque al GSL es la dimensión más crítica.

In [ ]:
# ─── Construcción del grafo de permisos ──────────────────────

def build_permission_graph(events: List[AdminEvent]) -> nx.DiGraph:
    """
    Grafo dirigido: actor → objeto con peso = impacto acumulado.
    Permite detectar patrones de escalada y ciclos anómalos.
    """
    G = nx.DiGraph()
    for evt in events:
        if evt.event_type in ('permission_granted','admin_role_granted',
                               'role_assigned','acl_modified'):
            w = IMPACT_WEIGHTS.get(evt.impact, 0.1)
            if G.has_edge(evt.actor_id, evt.object_id):
                G[evt.actor_id][evt.object_id]['weight'] += w
                G[evt.actor_id][evt.object_id]['count']  += 1
            else:
                G.add_edge(evt.actor_id, evt.object_id,
                           weight=w, count=1,
                           impact=evt.impact)
    return G


# ─── Vector de firma administrativo (8D) ─────────────────────

def signature_admin(
    events:     List[AdminEvent],
    actor_id:   str,
    cfg:        dict,
    graph:      Optional[nx.DiGraph] = None,
    baseline:   Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Vector de firma administrativa (8D) para un actor:
    [0] off_hours_ratio      — fracción de eventos fuera de ventana normal
    [1] critical_ratio       — fracción de eventos de impacto critical/high
    [2] sequence_violation   — violaciones de secuencia lógica detectadas
    [3] privilege_duration   — duración media de ventanas de privilegio elevado
    [4] object_diversity     — variedad de objetos afectados (cross-object spread)
    [5] gsl_touch_ratio      — fracción de eventos que tocan recursos GSL
    [6] failed_attempts      — fracción de intentos fallidos
    [7] graph_centrality     — centralidad del actor en el grafo de permisos
    """
    actor_evts = [e for e in events if e.actor_id == actor_id]
    if len(actor_evts) < 2:
        return np.full(8, 0.1, dtype=np.float32)

    adm_cfg = next((a for a in cfg['admins'] if a['id'] == actor_id), None)
    th      = cfg['thresholds']

    # [0] Off-hours ratio
    off_hours = 0
    for e in actor_evts:
        dt  = datetime.fromtimestamp(e.timestamp, tz=timezone.utc)
        win = NORMAL_WINDOWS.get(e.category, NORMAL_WINDOWS['account'])
        h_ok  = win['hours'][0] <= dt.hour <= win['hours'][1]
        d_ok  = dt.weekday() <= win['days'][1]
        if not (h_ok and d_ok):
            off_hours += 1
        # También verificar contra horario específico del actor
        if adm_cfg:
            ah_start, ah_end = adm_cfg['typical_hours']
            if not (ah_start <= dt.hour <= ah_end):
                off_hours += 0.5   # peso adicional
    off_ratio = float(off_hours / max(len(actor_evts), 1))

    # [1] Critical/high ratio
    crit = sum(1 for e in actor_evts
               if e.impact in ('critical', 'high'))
    crit_ratio = float(crit / len(actor_evts))

    # [2] Sequence violations
    violations = 0
    actor_types_seen = set()
    for e in sorted(actor_evts, key=lambda x: x.timestamp):
        prereqs = LOGICAL_SEQUENCE.get(e.event_type, [])
        if prereqs and not any(p in actor_types_seen for p in prereqs):
            violations += 1
        actor_types_seen.add(e.event_type)
    seq_viol = float(min(violations / max(len(actor_evts), 1), 1.0))

    # [3] Privilege duration
    sudo_starts = [e.timestamp for e in actor_evts
                   if e.event_type == 'sudo_session_start']
    sudo_ends   = [e.timestamp for e in actor_evts
                   if e.event_type == 'sudo_session_end']
    if sudo_starts and sudo_ends:
        durations = []
        for s in sudo_starts:
            ends_after = [e for e in sudo_ends if e > s]
            if ends_after:
                durations.append(min(ends_after) - s)
        max_dur = th.get('privilege_window_max_s', 3600)
        priv_dur = float(min(np.mean(durations) / max_dur, 1.0)) if durations else 0.0
    else:
        priv_dur = 0.0

    # [4] Object diversity
    objects    = set(e.object_id for e in actor_evts)
    obj_divers = float(min(len(objects) / 20.0, 1.0))

    # [5] GSL touch ratio
    gsl_res = set(cfg.get('gsl_protected_resources', []))
    gsl_touches = sum(1 for e in actor_evts
                      if e.object_id in gsl_res or e.object_type == 'gsl')
    gsl_ratio = float(gsl_touches / max(len(actor_evts), 1))

    # [6] Failed attempts
    failed    = sum(1 for e in actor_evts if not e.success)
    fail_rate = float(failed / max(len(actor_evts), 1))

    # [7] Graph centrality
    centrality = 0.0
    if graph and actor_id in graph:
        try:
            bc = nx.betweenness_centrality(graph)
            centrality = float(bc.get(actor_id, 0.0))
        except:
            centrality = float(graph.degree(actor_id)) / max(graph.number_of_nodes(), 1)

    vec = np.array([
        np.clip(off_ratio,  0, 1),
        np.clip(crit_ratio, 0, 1),
        np.clip(seq_viol,   0, 1),
        np.clip(priv_dur,   0, 1),
        np.clip(obj_divers, 0, 1),
        np.clip(gsl_ratio,  0, 1),
        np.clip(fail_rate,  0, 1),
        np.clip(centrality, 0, 1),
    ], dtype=np.float32)
    return vec


def compute_admin_dissonance(
    current:  np.ndarray,
    baseline: np.ndarray,
    cfg:      dict,
    events:   List[AdminEvent],
    actor_id: str,
) -> Tuple[float, List[str]]:
    """
    Disonancia administrativa con amplificadores contextuales.
    Devuelve (score, lista de razones).
    """
    th      = cfg['thresholds']
    gsl_res = set(cfg.get('gsl_protected_resources', []))

    # Pesos por dimensión: off_hours y gsl_touch son los más importantes
    weights = np.array([0.20, 0.15, 0.20, 0.10, 0.08, 0.15, 0.07, 0.05])
    weights /= weights.sum()
    diff     = np.abs(current - baseline)
    base_dis = float(np.dot(diff, weights))

    reasons   = []
    amplified = base_dis

    # Amplificador 1: cambios fuera de horario
    actor_evts = [e for e in events if e.actor_id == actor_id]
    off_hours  = sum(1 for e in actor_evts
                     if not (NORMAL_WINDOWS.get(e.category, {}).get('hours',(8,18))[0]
                             <= datetime.fromtimestamp(e.timestamp, tz=timezone.utc).hour
                             <= NORMAL_WINDOWS.get(e.category, {}).get('hours',(8,18))[1]))
    if off_hours > 0:
        amplified *= (1 + 0.3 * th.get('off_hours_weight', 2.5) *
                      off_hours / max(len(actor_evts), 1))
        reasons.append(f'{off_hours} eventos fuera de ventana horaria')

    # Amplificador 2: violaciones de secuencia lógica
    seq_violations = sum(1 for e in actor_evts
                         if current[2] > baseline[2] * 2)
    if current[2] > baseline[2] * 2:
        amplified *= (1 + 0.2 * th.get('sequence_violation_w', 3.0))
        reasons.append('Secuencia lógica de permisos violada')

    # Amplificador 3: toque a recursos GSL (autoataque)
    gsl_evts = [e for e in actor_evts
                if e.object_id in gsl_res or e.object_type == 'gsl']
    if gsl_evts:
        amplified *= (1 + th.get('gsl_self_attack_w', 5.0) *
                      len(gsl_evts) / max(len(actor_evts), 1))
        reasons.append(f'⚠️  {len(gsl_evts)} eventos sobre recursos GSL protegidos')

    return float(np.clip(amplified, 0, 1)), reasons


print('✅ Motor de manifold administrativo listo.')
print('   Dimensiones: off_hours · critical_ratio · seq_violation ·')
print('   priv_duration · obj_diversity · gsl_touch · fail_rate · centrality')

## 5. Motor de detección y registro forense

Procesa la secuencia de eventos, construye manifolds por actor,
detecta anomalías, y genera registros compatibles con el Modo 2.

In [ ]:
@dataclass
class AdminRecord:
    record_id:      str
    timestamp:      str
    run_id:         str
    actor_id:       str
    actor_role:     str
    dissonance:     float
    reasons:        List[str]
    # Dimensiones del vector
    dim_off_hours:  float
    dim_critical:   float
    dim_seq_viol:   float
    dim_priv_dur:   float
    dim_obj_divers: float
    dim_gsl_touch:  float
    dim_fail_rate:  float
    dim_centrality: float
    # Detalles del evento gatillo
    trigger_event:  str
    trigger_object: str
    trigger_obj_type: str
    gsl_self_attack: bool
    action:         str
    override_token: str
    sha256:         str = ''

    def compute_hash(self) -> str:
        payload = json.dumps({
            'record_id':  self.record_id,
            'timestamp':  self.timestamp,
            'actor_id':   self.actor_id,
            'dissonance': round(self.dissonance, 4),
            'gsl_attack': self.gsl_self_attack,
        }, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()[:16]

    def to_mode2_payload(self) -> dict:
        return {
            'event':          'admin_anomaly',
            'record_id':      self.record_id,
            'timestamp':      self.timestamp,
            'run_id':         self.run_id,
            'entity_id':      self.actor_id,
            'action':         self.action,
            'dissonance':     round(self.dissonance, 4),
            'executed':       False,
            'reversible':     True,
            'override_token': self.override_token,
            'ttl_minutes':    60,
            'metadata': {
                'reasons':         self.reasons,
                'gsl_self_attack': self.gsl_self_attack,
                'trigger_event':   self.trigger_event,
                'trigger_object':  self.trigger_object,
                'dims': {
                    'off_hours':  round(self.dim_off_hours,  3),
                    'critical':   round(self.dim_critical,   3),
                    'seq_viol':   round(self.dim_seq_viol,   3),
                    'priv_dur':   round(self.dim_priv_dur,   3),
                    'obj_divers': round(self.dim_obj_divers, 3),
                    'gsl_touch':  round(self.dim_gsl_touch,  3),
                    'fail_rate':  round(self.dim_fail_rate,  3),
                    'centrality': round(self.dim_centrality, 3),
                },
            },
            'sha256': self.sha256,
        }


FORENSIC_LOG_ADM: List[AdminRecord] = []

# Baselines por actor
_admin_baselines: Dict[str, np.ndarray] = {}


def run_admin_analysis(
    events:    List[AdminEvent],
    cfg:       dict,
    window_days: int  = 7,
    dry_run:   bool   = True,
) -> dict:
    """
    Analiza la secuencia completa de eventos administrativos:
    1. Construye el grafo de permisos
    2. Calcula vector de firma por actor en ventanas temporales
    3. Detecta anomalías y correlación con recursos GSL
    4. Genera registros forenses
    """
    global FORENSIC_LOG_ADM, _admin_baselines
    FORENSIC_LOG_ADM = []
    _admin_baselines = {}

    if not events:
        return {'records': [], 'graph': nx.DiGraph(), 'stats': {}}

    th      = cfg['thresholds']
    gsl_res = set(cfg.get('gsl_protected_resources', []))

    # Construir grafo completo
    graph = build_permission_graph(events)

    # Obtener actores únicos
    actors = list(set(e.actor_id for e in events))

    # Calcular baseline por actor (primeros 50% de eventos en orden temporal)
    events_sorted = sorted(events, key=lambda e: e.timestamp)
    split_idx = len(events_sorted) // 2
    baseline_events = events_sorted[:max(split_idx, 10)]

    for actor in actors:
        baseline_vec = signature_admin(
            baseline_events, actor, cfg, graph)
        _admin_baselines[actor] = baseline_vec

    # Analizar ventanas temporales deslizantes
    ts_min = events_sorted[0].timestamp
    ts_max = events_sorted[-1].timestamp
    window_s = window_days * 86400
    current  = ts_min + window_s

    while current <= ts_max + window_s:
        window_evts = [e for e in events_sorted
                       if current - window_s <= e.timestamp <= current]
        if not window_evts:
            current += window_s / 4
            continue

        for actor in actors:
            actor_evts = [e for e in window_evts if e.actor_id == actor]
            if len(actor_evts) < 2:
                continue

            vec = signature_admin(window_evts, actor, cfg, graph)
            baseline = _admin_baselines.get(actor, np.full(8, 0.1))
            dis, reasons = compute_admin_dissonance(
                vec, baseline, cfg, window_evts, actor)

            # Actualizar baseline gradualmente si no hay anomalía
            if dis < th['dissonance_alert']:
                alpha = 0.05
                _admin_baselines[actor] = (
                    (1-alpha)*baseline + alpha*vec)

            # Solo registrar si supera umbral
            if dis < th['dissonance_alert']:
                current += window_s / 4
                continue

            # Evento más grave en la ventana para este actor
            trigger = max(actor_evts,
                key=lambda e: IMPACT_WEIGHTS.get(e.impact, 0))
            gsl_attack = (
                trigger.object_id in gsl_res or
                trigger.object_type == 'gsl' or
                any(e.object_id in gsl_res for e in actor_evts)
            )

            # Acción
            if gsl_attack:
                action = 'alert_gsl_self_attack'
            elif dis >= th['dissonance_critical']:
                action = 'report_admin_critical'
            else:
                action = 'report_admin_anomaly'

            adm_cfg = next((a for a in cfg['admins']
                            if a['id'] == actor), {})
            ts_str  = datetime.fromtimestamp(
                trigger.timestamp, tz=timezone.utc).isoformat()
            token   = str(uuid.uuid4())[:12]

            rec = AdminRecord(
                record_id      = f'AR-{str(uuid.uuid4())[:8].upper()}',
                timestamp      = ts_str,
                run_id         = RUN_ID,
                actor_id       = actor,
                actor_role     = adm_cfg.get('role', 'unknown'),
                dissonance     = dis,
                reasons        = reasons,
                dim_off_hours  = float(vec[0]),
                dim_critical   = float(vec[1]),
                dim_seq_viol   = float(vec[2]),
                dim_priv_dur   = float(vec[3]),
                dim_obj_divers = float(vec[4]),
                dim_gsl_touch  = float(vec[5]),
                dim_fail_rate  = float(vec[6]),
                dim_centrality = float(vec[7]),
                trigger_event  = trigger.event_type,
                trigger_object = trigger.object_id,
                trigger_obj_type = trigger.object_type,
                gsl_self_attack= gsl_attack,
                action         = action,
                override_token = token,
            )
            rec.sha256 = rec.compute_hash()
            FORENSIC_LOG_ADM.append(rec)

        current += window_s / 4

    # Estadísticas
    gsl_atk = sum(1 for r in FORENSIC_LOG_ADM if r.gsl_self_attack)
    critical = sum(1 for r in FORENSIC_LOG_ADM
                   if r.dissonance >= th['dissonance_critical'])
    print(f'\n✅ Análisis administrativo completado')
    print(f'   Eventos procesados : {len(events)}')
    print(f'   Registros forenses : {len(FORENSIC_LOG_ADM)}')
    print(f'   Críticos           : {critical}')
    print(f'   Auto-ataques GSL   : {gsl_atk}')

    return {
        'records':   FORENSIC_LOG_ADM,
        'graph':     graph,
        'events':    events,
        'stats': {
            'total_events':  len(events),
            'forensic_recs': len(FORENSIC_LOG_ADM),
            'critical':      critical,
            'gsl_attacks':   gsl_atk,
            'actors':        list(set(e.actor_id for e in events)),
        }
    }


# ─── Demo ─────────────────────────────────────────────────────
print('⚙️  Generando escenario mixed...')
DEMO_EVENTS = generate_admin_events(ACFG, n_events=150,
                                     scenario='mixed', seed=42)
DEMO_RESULT = run_admin_analysis(DEMO_EVENTS, ACFG, dry_run=True)

## 6. Visualizaciones

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from PIL import Image
import io as io_module
pio.renderers.default = 'notebook'

DIM_LABELS = [
    'Fuera de horario', 'Alto impacto', 'Seq. violada',
    'Privilegio duración', 'Obj. diversidad',
    'Toque GSL', 'Intentos fallidos', 'Centralidad grafo'
]


def plot_dissonance_timeline(
    records: List[AdminRecord],
    cfg:     dict
) -> go.Figure:
    """Línea de tiempo de disonancia por actor con marcadores de auto-ataque."""
    if not records:
        return go.Figure()

    th_a = cfg['thresholds']['dissonance_alert']
    th_c = cfg['thresholds']['dissonance_critical']
    actors = list(set(r.actor_id for r in records))
    palette = ['#378ADD','#BA7517','#639922','#9B59B6','#D85A30']

    fig = go.Figure()
    for i, actor in enumerate(actors):
        recs = sorted([r for r in records if r.actor_id == actor],
                      key=lambda x: x.timestamp)
        x = [r.timestamp for r in recs]
        y = [r.dissonance for r in recs]
        texts = [f'{r.actor_id}<br>dis={r.dissonance:.3f}<br>'
                 f'{chr(10).join(r.reasons[:2])}' for r in recs]
        col = palette[i % len(palette)]

        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines+markers',
            name=actor,
            line=dict(color=col, width=1.5),
            marker=dict(size=7, color=col),
            text=texts,
            hovertemplate='%{text}<extra></extra>',
        ))

        # Marcar auto-ataques GSL
        gsl_recs = [r for r in recs if r.gsl_self_attack]
        if gsl_recs:
            fig.add_trace(go.Scatter(
                x=[r.timestamp for r in gsl_recs],
                y=[r.dissonance for r in gsl_recs],
                mode='markers',
                name=f'{actor} — AUTO-ATAQUE GSL',
                marker=dict(color='#E24B4A', size=14,
                            symbol='star', line=dict(color='white', width=1)),
                showlegend=True,
            ))

    fig.add_hline(y=th_a, line_dash='dash', line_color='#BA7517',
                  line_width=1,
                  annotation_text=f'alerta {th_a}',
                  annotation_font_size=9)
    fig.add_hline(y=th_c, line_dash='dash', line_color='#E24B4A',
                  line_width=1,
                  annotation_text=f'crítico {th_c}',
                  annotation_font_size=9)

    fig.update_layout(
        title='Disonancia administrativa por actor — línea de tiempo',
        xaxis_title='Timestamp', yaxis_title='Disonancia',
        yaxis_range=[0, 1.05], height=380,
        margin=dict(l=40, r=40, t=60, b=40),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=11),
        yaxis_gridcolor='#e8e8e4',
        legend=dict(orientation='h', y=-0.15),
    )
    return fig


def plot_dimension_radar(
    records: List[AdminRecord],
    actor_id: str
) -> go.Figure:
    """Radar de las 8 dimensiones del vector de firma para un actor."""
    actor_recs = [r for r in records if r.actor_id == actor_id]
    if not actor_recs:
        return go.Figure()

    # Promedio de dimensiones en registros anómalos
    dims = np.mean([[r.dim_off_hours, r.dim_critical, r.dim_seq_viol,
                     r.dim_priv_dur, r.dim_obj_divers, r.dim_gsl_touch,
                     r.dim_fail_rate, r.dim_centrality]
                    for r in actor_recs], axis=0)

    cats  = DIM_LABELS + [DIM_LABELS[0]]
    vals  = list(dims) + [dims[0]]
    b_val = [0.1] * 9

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=b_val, theta=cats,
        fill='toself',
        fillcolor='rgba(55,138,221,0.08)',
        line=dict(color='#378ADD', width=1, dash='dash'),
        name='Baseline',
    ))
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats,
        fill='toself',
        fillcolor='rgba(226,75,74,0.15)',
        line=dict(color='#E24B4A', width=2),
        name=f'{actor_id} — anomalía',
    ))
    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0,1],
                            tickfont_size=8)),
        title=f'Perfil de anomalía — {actor_id}',
        height=360,
        margin=dict(l=40, r=40, t=60, b=60),
        paper_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=10),
        legend=dict(orientation='h', y=-0.1),
    )
    return fig


def plot_permission_graph(
    graph: nx.DiGraph,
    gsl_resources: List[str]
) -> Image.Image:
    """Grafo de permisos con recursos GSL destacados."""
    if graph.number_of_nodes() == 0:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, 'Sin grafo', ha='center', va='center')
        buf = io_module.BytesIO()
        fig.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).copy()

    fig, ax = plt.subplots(figsize=(7, 5), dpi=100)
    fig.patch.set_facecolor('#f8f8f6')
    ax.set_facecolor('#f8f8f6')

    gsl_set = set(gsl_resources)
    admins  = set(ACFG['admins'][i]['id']
                  for i in range(len(ACFG['admins'])))

    node_colors = []
    for n in graph.nodes():
        if n in gsl_set:
            node_colors.append('#E24B4A')
        elif n in admins:
            node_colors.append('#378ADD')
        else:
            node_colors.append('#B5D4F4')

    pos = nx.spring_layout(graph, k=1.5, seed=42)
    weights = [graph[u][v].get('weight', 0.5) for u, v in graph.edges()]
    max_w   = max(weights) if weights else 1
    widths  = [1 + 3 * w / max_w for w in weights]

    nx.draw_networkx_nodes(graph, pos, ax=ax,
                           node_color=node_colors,
                           node_size=600, alpha=0.9)
    nx.draw_networkx_labels(graph, pos, ax=ax,
                            font_size=7, font_color='white',
                            font_weight='bold')
    nx.draw_networkx_edges(graph, pos, ax=ax,
                           width=widths,
                           edge_color='#9c9a92',
                           arrows=True, arrowsize=15,
                           alpha=0.6)

    handles = [
        mpatches.Patch(color='#378ADD', label='Actor (admin)'),
        mpatches.Patch(color='#E24B4A', label='Recurso GSL protegido'),
        mpatches.Patch(color='#B5D4F4', label='Objeto/recurso'),
    ]
    ax.legend(handles=handles, loc='lower left', fontsize=7,
              framealpha=0.8)
    ax.set_title('Grafo de permisos — aristas ponderadas por impacto',
                 fontsize=10)
    ax.axis('off')
    plt.tight_layout()

    buf = io_module.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()


# Mostrar visualizaciones del demo
from IPython.display import display
plot_dissonance_timeline(DEMO_RESULT['records'], ACFG).show()
if DEMO_RESULT['records']:
    plot_dimension_radar(
        DEMO_RESULT['records'],
        DEMO_RESULT['records'][0].actor_id
    ).show()
display(plot_permission_graph(
    DEMO_RESULT['graph'],
    ACFG['gsl_protected_resources']
))

## 7. Dashboard Gradio

In [ ]:
import gradio as gr

_last_result = DEMO_RESULT

SCENARIO_LABELS = {
    'normal':          'Normal — operación rutinaria',
    'priv_escalation': 'Escalada de privilegios (secuencia invertida)',
    'off_hours':       'Cambios críticos fuera de horario',
    'gsl_attack':      'Auto-ataque al GSL ⚠️',
    'slow_recon':      'Reconocimiento lento',
    'mixed':           'Mixto — normal + ataque',
}


def run_dashboard(
    scenario, n_events, window_days, dry_run_chk, seed, cfg_json
):
    global ACFG, _last_result

    try:
        custom = json.loads(cfg_json) if cfg_json.strip() else {}
        ACFG   = {**DEFAULT_ADMIN_CONFIG, **custom}
        for k in ('thresholds',):
            if k in DEFAULT_ADMIN_CONFIG and k in custom:
                ACFG[k] = {**DEFAULT_ADMIN_CONFIG[k], **custom[k]}
    except Exception as e:
        err = f'❌ Config inválida: {e}'
        return (None, None, None, err, '', '', '', '', '', [], [])

    events = generate_admin_events(
        ACFG, n_events=int(n_events),
        scenario=scenario, seed=int(seed)
    )
    result = run_admin_analysis(
        events, ACFG,
        window_days=int(window_days),
        dry_run=dry_run_chk
    )
    _last_result = result

    records = result['records']
    fig_tl  = plot_dissonance_timeline(records, ACFG)

    # Radar del actor con mayor disonancia
    if records:
        top_actor = max(records, key=lambda r: r.dissonance).actor_id
        fig_rad   = plot_dimension_radar(records, top_actor)
    else:
        fig_rad = go.Figure()

    fig_graph = plot_permission_graph(
        result['graph'], ACFG['gsl_protected_resources'])

    # Métricas
    s = result['stats']
    m_evts = str(s['total_events'])
    m_recs = str(s['forensic_recs'])
    m_crit = str(s['critical'])
    m_gsl  = str(s['gsl_attacks'])
    m_actors = ', '.join(s['actors'])

    # Tabla de registros forenses
    table = []
    for r in records[:50]:
        icon = '🚨' if r.gsl_self_attack else (
               '🔴' if r.dissonance >= ACFG['thresholds']['dissonance_critical']
               else '⚠️')
        table.append([
            r.record_id, r.actor_id, r.actor_role,
            f'{r.dissonance:.3f}', icon,
            r.trigger_event, r.trigger_object,
            '✓' if r.gsl_self_attack else '—',
            ' | '.join(r.reasons[:2]) if r.reasons else '—',
            r.override_token,
        ])

    # Tabla de eventos raw
    evts_table = []
    for e in sorted(events, key=lambda x: x.timestamp)[:60]:
        dt = datetime.fromtimestamp(e.timestamp, tz=timezone.utc)
        evts_table.append([
            dt.strftime('%Y-%m-%d %H:%M'),
            e.actor_id, e.event_type,
            e.object_id, e.impact,
            '✓' if e.success else '✗',
            e.source_ip,
        ])

    return (fig_tl, fig_rad, fig_graph,
            m_evts, m_recs, m_crit, m_gsl, m_actors,
            table, evts_table)


DEFAULT_CFG_UI = json.dumps({
    'org_name':    ACFG['org_name'],
    'iam_platform': 'generic',
    'thresholds':  ACFG['thresholds'],
    'gsl_protected_resources': ACFG['gsl_protected_resources'],
    'mode2_webhook_url': '',
}, indent=2)


with gr.Blocks(
    title='GSL Capa Administrativa',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate')
) as demo:

    gr.Markdown(f"""
# 🔑 GSL Capa Administrativa — Manifold de Comportamiento
**{ACFG['org_name']}** · Plataforma: {ACFG['iam_platform']}

Detecta anomalías en el comportamiento administrativo mediante firma geométrica.
El marcador ⭐ indica **auto-ataque al GSL** — intento de deshabilitar el sistema de detección.
""")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown('### ⚙️ Configuración')
            cfg_box = gr.Code(
                value=DEFAULT_CFG_UI, language='json',
                label='admin_config (editable)', lines=12)
            with gr.Row():
                scenario_dd = gr.Dropdown(
                    choices=list(SCENARIO_LABELS.keys()),
                    value='mixed',
                    label='Escenario',
                )
                n_evts_sl = gr.Slider(
                    50, 500, value=150, step=50,
                    label='Número de eventos')
            with gr.Row():
                win_sl = gr.Slider(
                    1, 30, value=7, step=1,
                    label='Ventana de análisis (días)')
                seed_sl = gr.Slider(
                    1, 999, value=42, step=1,
                    label='Semilla')
            dry_run_chk = gr.Checkbox(
                value=True,
                label='dry_run — no enviar al Modo 2')
            run_btn = gr.Button(
                '▶ Ejecutar análisis', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('### 📊 Métricas')
            with gr.Row():
                m_evts   = gr.Textbox(label='Eventos',      interactive=False)
                m_recs   = gr.Textbox(label='Registros',    interactive=False)
            with gr.Row():
                m_crit   = gr.Textbox(label='Críticos',     interactive=False)
                m_gsl    = gr.Textbox(label='Auto-ataques GSL ⭐', interactive=False)
            m_actors = gr.Textbox(label='Actores detectados', interactive=False)

            gr.Markdown("""
### 🔍 Dimensiones del vector de firma
| # | Dimensión | Detecta |
|---|---|---|
| 0 | Fuera de horario | Cambios en ventanas inusuales |
| 1 | Alto impacto | Densidad de eventos críticos |
| 2 | Seq. violada | Precondiciones omitidas |
| 3 | Privilegio dur. | Sesiones elevadas largas |
| 4 | Obj. diversidad | Spread de recursos afectados |
| 5 | Toque GSL | Auto-ataque al detector |
| 6 | Fallos | Intentos fallidos |
| 7 | Centralidad | Posición en grafo |
""")

    gr.Markdown('### 📈 Disonancia por actor — línea de tiempo')
    fig_tl_out = gr.Plot()

    with gr.Row():
        with gr.Column():
            gr.Markdown('### 🕸️ Radar de anomalía (actor con mayor disonancia)')
            fig_rad_out = gr.Plot()
        with gr.Column():
            gr.Markdown('### 🔗 Grafo de permisos')
            fig_graph_out = gr.Image(label='Grafo', type='pil')

    gr.Markdown('### 🗂️ Registro forense administrativo')
    table_out = gr.Dataframe(
        headers=['ID','Actor','Rol','Disonancia','Estado',
                 'Evento gatillo','Objeto','GSL attack',
                 'Razones','Token'],
        wrap=True
    )

    gr.Markdown('### 📋 Log de eventos administrativos (primeros 60)')
    evts_table_out = gr.Dataframe(
        headers=['Timestamp','Actor','Evento','Objeto',
                 'Impacto','Éxito','IP origen'],
        wrap=True
    )

    with gr.Accordion('📚 Integración con plataformas IAM reales', open=False):
        gr.Markdown("""
## Fuentes de log soportadas

### Active Directory (Windows Event Log)
Exportar con PowerShell:
```powershell
Get-WinEvent -LogName Security | Where-Object {$_.Id -in
  @(4720,4722,4725,4726,4728,4729,4732,4733,4670,
    4672,4673,4719,4756,4907,1102)} |
  Select-Object TimeCreated, Id, Message |
  ConvertTo-Json | Out-File ad_events.json
```

### Azure AD (Microsoft Graph API)
```python
import requests
headers = {'Authorization': f'Bearer {token}'}
r = requests.get(
  'https://graph.microsoft.com/v1.0/auditLogs/directoryAudits',
  headers=headers)
events = parse_admin_log(r.json()['value'], 'azure_ad', ACFG)
```

### Linux auditd
```bash
# Exportar eventos de los últimos 7 días
ausearch --start today -7 -i -l | \
  awk '/^type=SYSCALL/ {print}' | \
  python3 -c "import sys,json
for line in sys.stdin:
    fields = dict(kv.split('=',1)
      for kv in line.split() if '=' in kv)
    print(json.dumps(fields))" > auditd_events.jsonl
```

### CSV genérico
Cualquier CSV con columnas:
`timestamp, actor, event_type, object, ip, success`

### Recursos GSL a proteger
Agregar en `admin_config.json`:
```json
{"gsl_protected_resources": [
  "gsl_service_account",
  "gsl_policy",
  "gsl_audit_log",
  "resistor_rules",
  "manifold_store"
]}
```
Cualquier evento administrativo sobre estos recursos
activa el amplificador `gsl_self_attack_w` (×5.0 por defecto).
""")

    run_btn.click(
        fn=run_dashboard,
        inputs=[scenario_dd, n_evts_sl, win_sl,
                dry_run_chk, seed_sl, cfg_box],
        outputs=[fig_tl_out, fig_rad_out, fig_graph_out,
                 m_evts, m_recs, m_crit, m_gsl, m_actors,
                 table_out, evts_table_out]
    )

print('✅ Dashboard listo. Lanzando...')
demo.launch(share=False, inbrowser=True)

## 8. ExportPackage y persistencia

In [ ]:
import zipfile

# Registro forense
forensic_data = [r.to_mode2_payload() for r in FORENSIC_LOG_ADM]
with open(ARTIFACTS / 'forensic_log_admin.json', 'w') as f:
    json.dump(forensic_data, f, indent=2)

# Baselines de firma por actor
baselines_serial = {
    k: v.tolist() for k, v in _admin_baselines.items()
}
with open(ARTIFACTS / 'admin_signature_baselines.json', 'w') as f:
    json.dump(baselines_serial, f, indent=2)

# Grafo de permisos
graph_data = {
    'nodes': list(DEMO_RESULT['graph'].nodes()),
    'edges': [
        {'from': u, 'to': v, **d}
        for u, v, d in DEMO_RESULT['graph'].edges(data=True)
    ]
}
with open(ARTIFACTS / 'permission_graph.json', 'w') as f:
    json.dump(graph_data, f, indent=2)

# Manifiesto
manifest = {
    'run_id':           RUN_ID,
    'module':           'GSL-CapaAdministrativa',
    'mode2_compatible': True,
    'principal_id':     ACFG['principal_id'],
    'iam_platform':     ACFG['iam_platform'],
    'generated_at':     datetime.now(timezone.utc).isoformat(),
    'forensic_records': len(FORENSIC_LOG_ADM),
    'gsl_attacks':      sum(1 for r in FORENSIC_LOG_ADM if r.gsl_self_attack),
    'critical_records': sum(1 for r in FORENSIC_LOG_ADM
                            if r.dissonance >= ACFG['thresholds']['dissonance_critical']),
    'actors_monitored': list(_admin_baselines.keys()),
    'webhook_configured': bool(ACFG.get('mode2_webhook_url', '')),
}
with open(ARTIFACTS / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ZIP
export_zip = ARTIFACTS.parent.parent / \
    f'gsl_admin_{ACFG["principal_id"]}.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in ARTIFACTS.iterdir():
        zf.write(fpath, arcname=f'capa_admin/{fpath.name}')

print(f'\n✅ ExportPackage: {export_zip}')
print(f'   Registros forenses : {len(FORENSIC_LOG_ADM)}')
print(f'   Auto-ataques GSL   : {manifest["gsl_attacks"]}')
print(f'   Compatible Modo 2  : {manifest["mode2_compatible"]}')
print(f'   Tamaño             : {export_zip.stat().st_size / 1024:.1f} KB')
print(f'\n📋 Manifiesto:')
print(json.dumps(manifest, indent=2))